# BosesPH Fine-Tuning on Google Colab

This notebook fine-tunes a Whisper model on a BosesPH dataset using Colab's GPU. Generated by `bosesph export-colab`.

**Before you run:**
1. The dataset zip is downloaded automatically from your Google Drive
   share link (file ID `1kllfMNwiC1rA3dQlJmthLvG-iJgUcGf9`). Make sure the link
   is shared as **"Anyone with the link"**.
2. Set the runtime to GPU: **Runtime → Change runtime type → GPU**.
3. Run all cells (**Runtime → Run all**).

The trained model is zipped and downloaded to your machine at the end
(no Drive mount needed).

> If the repo `https://github.com/klydddd/asteria-classmates.git` is private, change the install cell to include a GitHub token, e.g. `git+https://<TOKEN>@github.com/...`.

In [ ]:
# Check the assigned GPU.
!nvidia-smi

In [ ]:
# Install the BosesPH toolkit (with training extras) from GitHub.
!pip install -q "git+https://github.com/klydddd/asteria-classmates.git@feat/colab-finetuning#egg=bosesph-toolkit[train]"
# Colab preinstalls an old torchao that peft rejects on import; we
# don't use torchao, so remove it to avoid an ImportError.
!pip uninstall -y torchao

In [ ]:
# === Training hyperparameters (edit as needed) ===
base_model = 'openai/whisper-small'
language = 'tl'
epochs = 3
max_steps = None
batch_size = 8
learning_rate = 1e-05
train_split = 'train'
eval_split = 'validation'
gradient_checkpointing = False
optim = 'adamw_torch'
use_lora = True
lora_r = 16
lora_alpha = 32
lora_dropout = 0.05
fp16 = True


In [ ]:
# Download the zipped dataset from Google Drive (no mount needed).
import glob
import os

!pip install -q gdown
import gdown

gdown.download(
    id='1kllfMNwiC1rA3dQlJmthLvG-iJgUcGf9',
    output="/content/dataset.zip",
    quiet=False,
)
!unzip -q -o /content/dataset.zip -d /content/data

# Locate the dataset folder (the one containing the train split CSV).
matches = glob.glob(
    f"/content/data/**/{train_split}.csv", recursive=True
)
assert matches, (
    f"{train_split}.csv not found in the zip — check its contents"
)
dataset_dir = os.path.dirname(matches[0])
output_dir = '/content/output'
print("dataset_dir =", dataset_dir)


In [ ]:
# Run fine-tuning on the GPU.
from bosesph.finetune import finetune_model

report = finetune_model(
    dataset_dir,
    output_dir,
    base_model=base_model,
    language=language,
    epochs=epochs,
    max_steps=max_steps,
    batch_size=batch_size,
    learning_rate=learning_rate,
    train_split=train_split,
    eval_split=eval_split,
    gradient_checkpointing=gradient_checkpointing,
    optim=optim,
    use_lora=use_lora,
    lora_r=lora_r,
    lora_alpha=lora_alpha,
    lora_dropout=lora_dropout,
    fp16=fp16,
    progress_fn=print,
)

print()
print("Fine-tuning complete.")
print("Model saved to:", report.model_path)
print("Config:", report.config_path)
print("Model card:", report.card_path)


In [ ]:
# Zip the fine-tuned model and download it to your machine.
import shutil

from google.colab import files

archive = shutil.make_archive("/content/finetuned_model", "zip", output_dir)
print("Created", archive)
files.download(archive)


## Done

The fine-tuned model was zipped and downloaded as
`finetuned_model.zip`. It contains:

- `model/` — the merged model + processor (load with
  `transformers.pipeline`).
- `model_card.md` — generated model card.
- `training_config.json` — the exact training configuration.

**Next steps:** unzip it locally, then evaluate it and run
`bosesph compare` against your baseline.
